# The Forecast Is Not the Fit

A source-frozen reconstruction of the **30 October 2026** House midterm chart. All observations are pre-2026. Gallup did not publish a comparable generic ballot for 1986, so that election is intentionally omitted. [Read the source ledger](data/PROVENANCE.md) and the [article](article.md).

In [ ]:
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory
from analysis import (ROOT, read_elections, read_scenario, fit_model, leave_one_out,
                      expanding_backtest, expanding_reduced_backtest, prediction_interval,
                      alternative_gallup, compute_evidence, save_tables)
from model_extensions import compute_model_evidence, save_model_evidence
from comparison_uncertainty import compute_paired_evidence, save_paired_evidence
from forecast_lock import build_forecast_lock, verify_lock, save_forecast_lock
from figures import generate_figures
rows = read_elections()
scenario = read_scenario()
assert len(rows) == 18 and 1986 not in {r.year for r in rows}
assert scenario.year == 2026 and scenario.party_margin == -8
print(f'Historical elections: {len(rows)}; 2026 generic ballot in GOP sign: {scenario.party_margin:+.1f}')

## First principles OLS and the missing 1986 row

For election $t$, $y_t$ is the change in seats held by the president's party, $m_t$ is its party-aligned generic-ballot margin, and $s_t$ is its pre-election seats defended. We fit $y_t=\beta_0+\beta_1m_t+\beta_2s_t+\epsilon_t$. No regression uses the 2026 election as an observed outcome.

In [ ]:
model = fit_model(rows)
print(f'R-squared = {model.r_squared:.3f}')
print(f'OLS coefficients [intercept, party-margin, seats] = {model.coefficients}')
assert 0.88 < model.r_squared < 0.90

In [ ]:
loo = leave_one_out(rows)
forward = expanding_backtest(rows, minimum_train=8)
loo_rmse = np.sqrt(np.mean([(r.predicted - r.actual)**2 for r in loo]))
forward_rmse = np.sqrt(np.mean([(r.predicted - r.actual)**2 for r in forward]))
baseline_rmse = np.sqrt(np.mean([(r.baseline - r.actual)**2 for r in forward]))
print(f'LOOCV RMSE {loo_rmse:.2f}; forward RMSE {forward_rmse:.2f}; forward baseline RMSE {baseline_rmse:.2f}')
assert len(forward) == 10 and forward_rmse < baseline_rmse

## Which predictor earns its place?

Compare the two-feature model with generic-ballot-only, defended-seats-only, and training-mean rules. Each prediction uses exactly the same preceding elections and is scored on the same ten target outcomes. This is a late-cycle, final-poll conditional comparison, not a live forecast comparison.

In [ ]:
reduced = expanding_reduced_backtest(rows, minimum_train=8)
assert [r.year for r in reduced] == [r.year for r in forward]
generic_rmse = np.sqrt(np.mean([(r.generic_only - r.actual)**2 for r in reduced]))
seats_rmse = np.sqrt(np.mean([(r.seats_only - r.actual)**2 for r in reduced]))
print(f'Two predictors: {forward_rmse:.2f} seats; generic-only: {generic_rmse:.2f}; seats-only: {seats_rmse:.2f}; mean: {baseline_rmse:.2f}')
assert forward_rmse < generic_rmse < seats_rmse < baseline_rmse

In [ ]:
point = model.predict(scenario.party_margin, scenario.seats)
interval = prediction_interval(model, scenario.party_margin, scenario.seats, level=0.95)
alternative = fit_model(alternative_gallup(rows))
print(f'2026 GOP seat change: {point:.2f}; conditional 95% PI: ({interval[0]:.2f}, {interval[1]:.2f})')
print(f'Alternative Gallup retrospective: R²={alternative.r_squared:.3f}, forecast={alternative.predict(scenario.party_margin, scenario.seats):.2f}')
assert -34 < point < -31 and interval[0] < point < interval[1]

## Three additional model assumptions, one fixed historical test

Compare ordinary least squares with a fixed-four-degrees-of-freedom Student-t regression, ridge regularization of standardized slopes (alpha=2), and a recency-weighted least-squares fit (24-year half-life). None of these hyperparameters is chosen by the ten historical test errors. The models share the same forward targets and only use each target's earlier elections. The historical origin path evaluates the fixed **2026 input** with progressively longer historical training samples; it is **not** an archive of forecasts made in those years.

In [ ]:
candidate_evidence = compute_model_evidence()
assert [r.year for r in candidate_evidence.folds] == [r.year for r in forward]
for candidate in ('ols', 'student_t', 'ridge', 'recency_wls'):
    primary = candidate_evidence.summary['primary'][candidate]
    alternate = candidate_evidence.summary['alternative_gallup'][candidate]
    print(f'{candidate:12s} RMSE {primary["rmse"]:.2f}; alternate Gallup RMSE {alternate["rmse"]:.2f}; 2026 scenario {primary["forecast_2026"]:+.2f}')
assert len(candidate_evidence.folds) == 10
assert len(candidate_evidence.coefficient_path) == 11

## The uncertainty of the model ranking

Compare each alternative against OLS **within the same ten held-out years**. One diagnostic drops one target election at a time. A second draws 20,000 paired year resamples (seed 20261031) and recomputes candidate-minus-OLS RMSE. These paired-bootstrap percentiles are exploratory: they are **not calibrated confidence intervals**, because electoral cycles need not be exchangeable and ten observations cannot establish a stable error distribution.

In [ ]:
paired_evidence = compute_paired_evidence(candidate_evidence)
assert len(paired_evidence.losses) == 20 and len(paired_evidence.comparisons) == 6
for result in paired_evidence.comparisons:
    print(f'{result.source:18s} {result.candidate:11s} ΔRMSE={result.delta_rmse:+.2f}; resampled percentiles=({result.bootstrap_q025:+.2f}, {result.bootstrap_q975:+.2f}); omitted-year reversals={result.omitted_year_reversal_count}')
assert next(r for r in paired_evidence.comparisons if r.source == 'primary' and r.candidate == 'student_t').omitted_year_reversal_count == 1

## Registration before the 2026 result

The model evidence uses a **31 October information cut-off** and is separately **registered on 1 November**, before the 3 November election. The content-addressed manifest records the point forecast, conditional interval, alternative point estimates and source SHA-256 fingerprints. No observed 2026 result is present. A later result must be scored separately against this unchanged record.

In [ ]:
locked_forecast = build_forecast_lock()
verify_lock(locked_forecast, current_root=ROOT)
assert locked_forecast['target']['observed_2026_outcome'] is None
assert locked_forecast['forecast_information_as_of'] == '2026-10-31'
assert locked_forecast['registration_date'] == '2026-11-01'
print(f'Registered {locked_forecast["primary_forecast"]["predicted_seat_change"]:+.2f} seats, before the {locked_forecast["general_election_date"]} election')

In [ ]:
evidence = compute_evidence(draws=5000, seed=20261031)
with TemporaryDirectory(prefix='midterm-study-') as temp:
    target = Path(temp)
    save_tables(evidence, target / 'results')
    save_model_evidence(candidate_evidence, target / 'results')
    save_paired_evidence(paired_evidence, target / 'results')
    save_forecast_lock(locked_forecast, target / 'results' / 'forecast_lock.json')
    paths = generate_figures(evidence, target / 'figures')
    assert len(paths) == 13 and (target / 'results' / 'summary.json').is_file()
    assert (target / 'results' / 'alternative_models_summary.json').is_file()
    assert (target / 'results' / 'paired_uncertainty_summary.json').is_file()
    assert (target / 'results' / 'forecast_lock.json').is_file()
print(f'Regenerated {len(paths)} figures and model evidence in a clean temporary directory')

## What is and is not validated

The historical fit, leave-one-out predictions, genuinely forward-in-time predictions and conditional forecast interval are different quantities. The forecast is not an observation, the interval assumes stable errors, and there is no causal identification. Re-running an observed poll average at the end of a historical cycle is not the same as predicting that cycle months in advance.